# Raman Spectral Classification — Pipeline Results

This notebook visualizes all results from `run_full_pipeline.py`.

**Sections:**
1. Data summary and cleaning report
2. Model leaderboard comparison
3. Confusion matrices
4. Per-class F1 analysis
5. Ensemble results


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', palette='deep')
%matplotlib inline

# Find the latest pipeline run
output_root = Path('../outputs/pipeline_runs')
runs = sorted(output_root.glob('run_*')) if output_root.exists() else []
if runs:
    RUN_DIR = runs[-1]
    print(f'Using latest run: {RUN_DIR.name}')
else:
    print('No pipeline runs found. Run: python experiments/run_full_pipeline.py')
    RUN_DIR = None

## 1. Data Summary & Cleaning

In [ ]:
if RUN_DIR:
    cleaning = json.loads((RUN_DIR / 'cleaning_report.json').read_text())
    data_summary = json.loads((RUN_DIR / 'data_summary.json').read_text())

    print(f"Classes: {data_summary['num_classes']}")
    print(f"Labels: {data_summary['label_names']}")
    print()
    
    rows = []
    for split, info in cleaning.items():
        rows.append({'Split': split, 'Original': info['original'], 'Cleaned': info['cleaned'],
                     'Removed': info['original'] - info['cleaned']})
    display(pd.DataFrame(rows))
    
    print(f"\nSplits: {data_summary['splits']}")

## 2. Model Leaderboard

In [ ]:
if RUN_DIR and (RUN_DIR / 'leaderboard.csv').exists():
    lb = pd.read_csv(RUN_DIR / 'leaderboard.csv')
    display(lb[['model_name', 'model_group', 'val_accuracy', 'val_macro_f1']].style
            .format({'val_accuracy': '{:.4f}', 'val_macro_f1': '{:.4f}'})
            .background_gradient(subset=['val_accuracy'], cmap='Greens'))
    
    if 'test_accuracy' in lb.columns:
        print('\nTest Accuracy:')
        display(lb[['model_name', 'test_accuracy']].dropna().style
                .format({'test_accuracy': '{:.4f}'})
                .background_gradient(subset=['test_accuracy'], cmap='Greens'))

In [ ]:
if RUN_DIR and (RUN_DIR / 'leaderboard.csv').exists():
    lb = pd.read_csv(RUN_DIR / 'leaderboard.csv')
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#2ecc71' if g == 'ensemble' else '#3498db' if g == 'deep' else '#e67e22'
              for g in lb['model_group']]
    bars = ax.barh(lb['model_name'], lb['val_accuracy'], color=colors, edgecolor='white')
    ax.set_xlabel('Validation Accuracy')
    ax.set_title('Model Comparison — Validation Accuracy')
    ax.set_xlim(left=max(0, lb['val_accuracy'].min() - 0.05))
    for bar, acc in zip(bars, lb['val_accuracy']):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{acc:.4f}', va='center', fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 3. Confusion Matrices

In [ ]:
if RUN_DIR:
    # Show confusion matrices for top models on test split
    cm_paths = list(RUN_DIR.rglob('test/confusion_matrix.png'))
    if cm_paths:
        n = min(4, len(cm_paths))
        fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
        if n == 1:
            axes = [axes]
        for ax, path in zip(axes, cm_paths[:n]):
            img = plt.imread(str(path))
            ax.imshow(img)
            ax.set_title(path.parent.parent.name, fontsize=10)
            ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print('No test confusion matrices found.')

## 4. Per-Class F1 Analysis

In [ ]:
if RUN_DIR:
    # Find the best model's test metrics
    metric_files = list(RUN_DIR.rglob('test/metrics.json'))
    if metric_files:
        best_metrics = json.loads(metric_files[0].read_text())
        if 'classification_report' in best_metrics:
            report = best_metrics['classification_report']
            classes = [k for k in report.keys() if k not in ('accuracy', 'macro avg', 'weighted avg')]
            f1s = [report[c]['f1-score'] for c in classes]
            
            fig, ax = plt.subplots(figsize=(12, 5))
            bars = ax.bar(classes, f1s, color='#3498db', edgecolor='white')
            ax.axhline(y=0.97, color='red', linestyle='--', alpha=0.5, label='97% target')
            ax.set_ylabel('F1-Score')
            ax.set_title(f'Per-Class F1 Score — {metric_files[0].parent.parent.name}')
            ax.set_ylim(0, 1.05)
            ax.legend()
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
    else:
        print('No test metrics found.')

## 5. Ensemble Results

In [ ]:
if RUN_DIR:
    ensemble_dir = RUN_DIR / 'ensemble'
    if ensemble_dir.exists() and (ensemble_dir / 'summary_metrics.json').exists():
        ens_metrics = json.loads((ensemble_dir / 'summary_metrics.json').read_text())
        print('=== Ensemble Summary ===')
        for split, m in ens_metrics.items():
            print(f"  {split}: acc={m.get('accuracy',0):.4f}  macro_f1={m.get('macro_f1',0):.4f}")
        
        if (ensemble_dir / 'ensemble_config.json').exists():
            ens_cfg = json.loads((ensemble_dir / 'ensemble_config.json').read_text())
            print('\nMembers:')
            for m in ens_cfg.get('members', []):
                print(f"  - {m['model_name']}")
    else:
        print('No ensemble results found.')